# Train the basic NAT decision tree

This intentionally small model uses exactly two raw flow fields: `IP_TTL` and `DST_PORT`. It aggregates each field to its number of unique values for one source IP in one 15-minute window, then tunes the decision-tree depth over 3, 4, and 5.

In [ ]:
from pathlib import Path
import io
import sys

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").exists()
)
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from detectors.common import TimeWindowedIPFlowDataset
from detectors.nat_detector import NATClassifier

In [ ]:
training_dir = repo_root / "src/detectors/nat_detector/training"
data_path = training_dir / "data/combined_NAT_anonym.csv"
model_path = (
    repo_root
    / "src/detectors/nat_detector/final_models/basic_tree"
    / "nat_basic_tree.joblib"
)
rows_per_class = 100_000
window_size = "15min"
random_state = 42
max_depth_values = [3, 4, 5]

aggregation = {
    "unique_ttl_values": ("IP_TTL", "nunique"),
    "unique_destination_ports": ("DST_PORT", "nunique"),
}
classifier = NATClassifier(
    DecisionTreeClassifier(
        class_weight="balanced",
        random_state=random_state,
    ),
    aggregation,
)

Read file.

In [ ]:
non_nat_flows = pd.read_csv(
    data_path_no_nat,
    usecols=columns_to_load,
)
nat_flows = pd.read_csv(
    data_path_nat,
    usecols=columns_to_load,
)

label_values = {
    "false": 0,
    "0": 0,
    "true": 1,
    "1": 1,
}
for frame in (non_nat_flows, nat_flows):
    normalized_labels = frame[label_column].astype(str).str.strip().str.lower()
    frame[label_column] = normalized_labels.map(label_values).astype("int8")

if not non_nat_flows[label_column].eq(0).all():
    raise ValueError("The first sampled rows are not exclusively non-NAT")
if not nat_flows[label_column].eq(1).all():
    raise ValueError("The last sampled rows are not exclusively NAT")

sampled_flows = pd.concat([non_nat_flows, nat_flows], ignore_index=True)
sampled_flows[label_column].value_counts().sort_index()

In [ ]:
columns = ["SRC_IP", "TIME_FIRST", "IP_TTL", "DST_PORT", "IS_NAT"]
non_nat_flows = pd.read_csv(data_path, usecols=columns, nrows=rows_per_class)
nat_flows = read_last_csv_rows(data_path, rows_per_class, columns)

label_values = {"false": 0, "0": 0, "true": 1, "1": 1}
for frame in (non_nat_flows, nat_flows):
    labels = frame["IS_NAT"].astype(str).str.strip().str.lower()
    frame["IS_NAT"] = labels.map(label_values).astype("int8")
if not non_nat_flows["IS_NAT"].eq(0).all():
    raise ValueError("The first sampled rows are not exclusively non-NAT")
if not nat_flows["IS_NAT"].eq(1).all():
    raise ValueError("The last sampled rows are not exclusively NAT")
flows = pd.concat([non_nat_flows, nat_flows], ignore_index=True)
flows["IS_NAT"].value_counts().sort_index()

Aggregate with the exact classifier method used at runtime, verify labels per IP/window, and balance the resulting model samples.

In [ ]:
dataset = TimeWindowedIPFlowDataset(
    flows,
    timestamp_column="TIME_FIRST",
    window_size=window_size,
)
X = dataset.apply(classifier.aggregate)
label_summary = pd.concat(
    {
        window_start: ip_dataset.agg(
            is_nat=("IS_NAT", "first"),
            distinct_labels=("IS_NAT", "nunique"),
        )
        for window_start, ip_dataset in dataset
    },
    names=["window_start", "SRC_IP"],
)
if not label_summary["distinct_labels"].eq(1).all():
    raise ValueError("An IP/window contains conflicting NAT labels")

table = X.join(label_summary["is_nat"])
samples_per_class = int(table["is_nat"].value_counts().min())
table = (
    table.groupby("is_nat", group_keys=False)
    .sample(n=samples_per_class, random_state=random_state)
    .sort_index()
)
table["is_nat"].value_counts().sort_index()

In [ ]:
X = table.drop(columns="is_nat")
y = table["is_nat"].astype("int8")
groups = X.index.get_level_values("SRC_IP")
splitter = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=random_state
)
train_positions, test_positions = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_positions], X.iloc[test_positions]
y_train, y_test = y.iloc[train_positions], y.iloc[test_positions]

train_groups = X_train.index.get_level_values("SRC_IP")
parameter_search = GridSearchCV(
    estimator=classifier.model,
    param_grid={"max_depth": max_depth_values},
    scoring="f1",
    cv=StratifiedGroupKFold(
        n_splits=5, shuffle=True, random_state=random_state
    ),
    n_jobs=-1,
)
parameter_search.fit(X_train, y_train, groups=train_groups)
classifier.model = parameter_search.best_estimator_
tuning_results = (
    pd.DataFrame(parameter_search.cv_results_)
    .loc[:, ["param_max_depth", "mean_test_score", "std_test_score"]]
    .sort_values("param_max_depth")
    .reset_index(drop=True)
)
print(f"Selected max_depth: {classifier.model.max_depth}")
display(tuning_results)

predictions = classifier.model.predict(X_test)
metrics = pd.Series(
    {
        "accuracy": accuracy_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
    },
    name="score",
)
confusion = pd.DataFrame(
    confusion_matrix(y_test, predictions, labels=[0, 1]),
    index=["actual_non_nat", "actual_nat"],
    columns=["predicted_non_nat", "predicted_nat"],
)
display(metrics.to_frame(), confusion)

In [ ]:
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(classifier, model_path)
loaded_classifier = joblib.load(model_path)
print(f"Saved model: {model_path}")